In [1]:
import torch
from PIL import Image
import open_clip

/Users/zhiweizhang/Projects/nazi_symbols_classification/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [3]:
from nazi_symbols_classification.training.data_preparation import get_image_paths

In [4]:
images = get_image_paths("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-detection", 
                         ("train", "test", "valid"))

In [9]:
import os

non_nazi_train_extra = [os.path.join("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/train", file_name)
                        for file_name in os.listdir("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/train")
                        if not file_name.startswith(".")]
non_nazi_test_extra = [os.path.join("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/test", file_name)
                        for file_name in os.listdir("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/test")
                        if not file_name.startswith(".")]
non_nazi_val_extra = [os.path.join("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/val", file_name)
                        for file_name in os.listdir("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/val")
                        if not file_name.startswith(".")]

In [10]:
train_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-detection/train')]
test_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-detection/test')]
valid_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-detection/valid')]

In [11]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [12]:
train_images += non_nazi_train_extra
y_train += ["non-nazi"] * len(non_nazi_train_extra)

In [13]:
test_images += non_nazi_test_extra
y_test += ["non-nazi"] * len(non_nazi_test_extra)

In [14]:
valid_images += non_nazi_val_extra
y_valid += ["non-nazi"] * len(non_nazi_val_extra)

In [6]:
import numpy as np
import pandas as pd


def load_image(image_path):
    with torch.no_grad(), torch.cpu.amp.autocast():
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [47]:
from sklearn.naive_bayes import MultinomialNB

In [63]:
mnb = MultinomialNB().fit(training_data, y_train[:1000])

ValueError: Negative values in data passed to MultinomialNB (input X)

In [102]:
with open("training_data.csv", "w") as f:
    f.write(",".join(map(str, range(len(train_images)))))

for i in range(0, len(train_images), 200):
    data = preprocess_images(train_images[i:i+200])
    with open("training_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [104]:
with open("validation_data.csv", "w") as f:
    f.write(",".join(map(str, range(len(valid_images)))))

for i in range(0, len(valid_images), 200):
    data = preprocess_images(valid_images[i:i+200])-'
    with open("validation_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [105]:
with open("test_data.csv", "w") as f:
    f.write(",".join(map(str, range(len(test_images)))))

for i in range(0, len(test_images), 200):
    data = preprocess_images(test_images[i:i+200])
    with open("test_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [7]:
training_data = pd.read_csv("training_data.csv")
validation_data = pd.read_csv("validation_data.csv")
test_data = pd.read_csv("test_data.csv")

In [137]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data, y_train)

print("score on test: " + str(lr.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9895097841436352
CPU times: user 1.33 s, sys: 673 ms, total: 2 s
Wall time: 134 ms


In [138]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data, y_train)

print("score on test: " + str(sgd.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9890054468428485
CPU times: user 294 ms, sys: 104 ms, total: 398 ms
Wall time: 115 ms


In [15]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data, y_train)

print("score on test: " + str(knn.score(pd.concat([validation_data, test_data]), y_valid + y_test)))


Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md



score on test: 0.9908210611256808
CPU times: user 7.27 s, sys: 238 ms, total: 7.51 s
Wall time: 579 ms


In [140]:
%%time

# import the library
from sklearn.svm import LinearSVC

# instantiate & fit
svm=LinearSVC(C=0.0001)
svm.fit(training_data, y_train)

print("score on test: " + str(svm.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9503732096025822
CPU times: user 212 ms, sys: 229 ms, total: 441 ms
Wall time: 172 ms


In [141]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data, y_train)

print("score on test: " + str(clf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9690336897316926
CPU times: user 763 ms, sys: 13.5 ms, total: 777 ms
Wall time: 776 ms


In [142]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data, y_train)

print("score on test: " + str(bg.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9634859794230382
CPU times: user 3.52 s, sys: 73.6 ms, total: 3.6 s
Wall time: 3.6 s


In [143]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data, y_train)

print("score on test: " + str(adb.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


score on test: 0.9888037119225338
CPU times: user 49 s, sys: 536 ms, total: 49.6 s
Wall time: 49.6 s


In [144]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data, y_train)

print("score on test: " + str(gbc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9886019770022191
CPU times: user 1min 11s, sys: 204 ms, total: 1min 11s
Wall time: 1min 11s


In [145]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data, y_train)

print("score on test: " + str(rf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9823481944724631
CPU times: user 7.15 s, sys: 198 ms, total: 7.35 s
Wall time: 7.35 s


In [146]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier

# 2) logistic regression =lr
lr=LogisticRegression(max_iter=5000)
# 3) random forest =rf
rf = RandomForestClassifier(n_estimators=30,max_depth=3)
# 4) suport vecotr mnachine = svm
svm=LinearSVC(max_iter=5000)
evc=VotingClassifier(estimators=[('lr',lr),('rf',rf),('svm',svm)])
evc.fit(training_data, y_train)

print("score on test: " + str(evc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.9892071817631632
CPU times: user 2.98 s, sys: 2.58 s, total: 5.56 s
Wall time: 1.44 s


In [147]:
len(training_images), len(valid_images) + len(test_images)

(6561, 9914)

In [17]:
import os

images = [os.path.join("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test", file_name)
            for file_name in os.listdir("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test")
            if not file_name.startswith(".")]

In [18]:
len(images)

91110

In [19]:
images = sorted(images)

In [21]:
knn.predict(load_image(images[0]))[0]

X does not have valid feature names, but KNeighborsClassifier was fitted with feature names


'non-nazi'

In [ ]:
result = [knn.predict(load_image(image))[0] for image in images]

X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
X does not have valid feature names, but KNeighborsClassifier was fitted with fe

In [ ]:
import json

with open('new_test_result.json', 'w') as f:
    json.dump(result, f)

In [26]:
print("hello")

hello


In [27]:
len(result)

91110

In [28]:
result.count('non-nazi')

83671

In [30]:
result.count('nazi-symbol')

7439

In [32]:
wrong_images = [images[i] for i, pred_result in enumerate(result) if pred_result == 'nazi-symbol']

In [33]:
wrong_images

['/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/00059a760c0e4423a3bf55ea0bfc7480.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/000ff55056384d61972bda1b6bd80701.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/0011c570133b4117b735ef0eafde3343.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/0014d21f66fc4946a2d0258182cd7e42.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/0019800497824bbfb7635b764ac07d37.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/001fc748e6.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/002bb8e03b.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/new_test/002de5119a454775af944cf8606fbe4f.jpg',
 '/Users/zhiweizhang/Projects/nazi_symbols_c

In [34]:
kids_and_adults_images = get_image_paths("/Users/zhiweizhang/.cache/kagglehub/datasets/kaushigihanml/kids-and-adults-detection/versions/2/children_and_adults/images", 
                                         sub_folders=("train", "val"))
scooby_doo_images = get_image_paths("/Users/zhiweizhang/.cache/kagglehub/datasets/esmanurdeli/scooby-doo-classification-dataset/versions/1/dataset", 
                                    sub_folders=None)
low_light_animal_images = get_image_paths("/Users/zhiweizhang/.cache/kagglehub/datasets/subratasarkar32/low-light-animals/versions/2/animals_low_light/animals_low_light",
                                          sub_folders=None)
ai_humen_generated_images = get_image_paths("/Users/zhiweizhang/.cache/kagglehub/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4",
                                         sub_folders=None)

In [35]:
image_path_to_cls = dict()
for image in kids_and_adults_images:
    image_path_to_cls[os.path.basename(image)] = "kids_and_adults_images"
for image in scooby_doo_images:
    image_path_to_cls[os.path.basename(image)] = "scooby_doo_images"
for image in low_light_animal_images:
    image_path_to_cls[os.path.basename(image)] = "low_light_animal_images"
for image in ai_humen_generated_images:
    image_path_to_cls[os.path.basename(image)] = "ai_humen_generated_images"

In [36]:
wrong_image_cls = []
for image in wrong_images:
    wrong_image_cls.append(image_path_to_cls[os.path.basename(image)])

In [43]:
wrong_image_cls.count("kids_and_adults_images"), len(kids_and_adults_images)

(0, 0)

In [44]:
wrong_image_cls.count("scooby_doo_images"), len(scooby_doo_images)

(164, 221)

In [45]:
wrong_image_cls.count("low_light_animal_images"), len(low_light_animal_images)

(1001, 5400)

In [46]:
wrong_image_cls.count("ai_humen_generated_images"), len(ai_humen_generated_images)

(6274, 85490)